## Cell 1 — clone + install (idempotent)

Pulls the latest code from GitHub, installs dependencies into the Kaggle Python env, and registers the Atari ROMs. Safe to re-run.

In [ ]:
import os, shutil, sys

REPO_URL  = 'https://github.com/<your-username>/atari-solaris.git'  # <-- EDIT THIS
BRANCH    = 'main'
REPO_DIR  = '/kaggle/working/atari-solaris'
CKPT_DIR  = f'{REPO_DIR}/checkpoints'

# Stash checkpoints/ before clone so a re-run never nukes saved weights.
_STASH = '/kaggle/working/_ckpt_stash'
if os.path.isdir(_STASH):
    shutil.rmtree(_STASH)
if os.path.isdir(f'{REPO_DIR}/checkpoints'):
    shutil.move(f'{REPO_DIR}/checkpoints', _STASH)
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)

os.chdir('/kaggle/working')
!git clone --branch {BRANCH} {REPO_URL}
os.chdir(REPO_DIR)
!git pull origin {BRANCH} || echo 'pull skipped'

# Restore stashed checkpoints (fresh clone has none, so this just moves them back).
if os.path.isdir(_STASH):
    if os.path.isdir(f'{REPO_DIR}/checkpoints'):
        shutil.rmtree(f'{REPO_DIR}/checkpoints')
    shutil.move(_STASH, f'{REPO_DIR}/checkpoints')
    print('[resume] restored checkpoint directory')
else:
    os.makedirs(f'{REPO_DIR}/checkpoints', exist_ok=True)

# Kaggle already ships PyTorch + CUDA — skip reinstalling torch to avoid
# breaking the preconfigured CUDA linkage.
!pip install -q ale-py 'gymnasium[accept-rom-license]' AutoROM \
    opencv-python-headless tensorboard pyyaml imageio imageio-ffmpeg
!AutoROM --accept-license -y > /dev/null 2>&1

import gymnasium as gym, ale_py
gym.register_envs(ale_py)
env = gym.make('ALE/Solaris-v5', frameskip=1)
env.close()
print('envs registered OK')

## Cell 2 — CONFIG

Set which experiment to run. `exp_name` is derived automatically from the YAML so all subsequent cells stay in sync.

In [ ]:
import yaml

CONFIG_PATH = 'configs/double_dqn.yaml'  # or 'configs/dqn_baseline.yaml'
SEED        = 616

_cfg     = yaml.safe_load(open(CONFIG_PATH, encoding='utf-8'))
exp_name = f"{_cfg['experiment']['name']}_seed{SEED}"
del _cfg
print(f'exp_name = {exp_name}')

## Cell 3 — smoketest (~3 min on T4)

Runs 5,000 training steps to validate the full pipeline before committing to the multi-hour run. Expected: a `.pt` saved under `checkpoints/`, TensorBoard events written, no NaNs.

Set `SMOKETEST = False` to skip once it has passed in a prior session.

In [ ]:
import os
os.chdir(REPO_DIR)

SMOKETEST = True
if SMOKETEST:
    !python -u train.py --config {CONFIG_PATH} --seed {SEED} --total-steps 5000

    exp_dir = os.path.join(CKPT_DIR, exp_name)
    ckpts   = [f for f in os.listdir(exp_dir) if f.endswith('.pt')]
    tb_logs = os.path.join(exp_dir, 'tb_logs')

    assert ckpts,              f'no checkpoint saved under {exp_dir}'
    assert os.path.isdir(tb_logs) and os.listdir(tb_logs), \
        f'no TensorBoard events under {tb_logs}'

    # Wipe the 5K-step checkpoint so the real run starts clean
    import shutil
    shutil.rmtree(exp_dir)
    print('\n[smoketest] OK — checkpoint dir wiped, ready for full run')
else:
    print('[smoketest] skipped')

## Cell 4 — full 3,000,000-step run on T4

Wall-time estimate on T4: ~4–5 hours (one Kaggle session). Saves a `.pt` every 25K steps, runs 5 greedy eval episodes every 25K steps. Logs go to `checkpoints/<exp_name>/tb_logs/`.

If the session disconnects, re-run the whole notebook — Cell 1 restores checkpoints, Cell 4 resumes automatically from the latest `step_*.pt`.

In [ ]:
import os
os.chdir(REPO_DIR)
LOG_PATH = os.path.join(CKPT_DIR, exp_name, 'train.log')
os.makedirs(os.path.dirname(LOG_PATH), exist_ok=True)
!python -u train.py --config {CONFIG_PATH} --seed {SEED} --resume 2>&1 | tee {LOG_PATH}

## Cell 5 — greedy evaluation (10 episodes)

Loads `final.pt` and runs the trained agent for 10 full episodes with ε=0.05. Reports mean ± std true game score (unclipped). Compare against: random baseline ~2670, Rainbow@200M ~3560, human ~12327.

In [ ]:
import os
os.chdir(REPO_DIR)
FINAL_CKPT = os.path.join(CKPT_DIR, exp_name, 'final.pt')
assert os.path.exists(FINAL_CKPT), f'final.pt not found at {FINAL_CKPT}; has Cell 4 finished?'

import sys, torch, numpy as np
sys.path.insert(0, REPO_DIR)
from agents.dqn import DQNAgent
from env.wrappers import make_eval_env

ckpt      = torch.load(FINAL_CKPT, map_location='cpu', weights_only=False)
hp        = ckpt.get('hp', {})
agent_cfg = hp.get('agent', {})
agent = DQNAgent(
    n_actions=18, obs_shape=(4, 84, 84), device='cpu',
    lr=float(agent_cfg.get('lr', 1e-4)),
    gamma=float(agent_cfg.get('gamma', 0.99)),
    buffer_capacity=1000,   # dummy — not used for eval
    batch_size=32,
    learning_starts=999,
    target_update_freq=int(agent_cfg.get('target_update_freq', 10000)),
    eps_start=0.0, eps_end=0.0, eps_decay_steps=1,
    double_dqn=bool(agent_cfg.get('double_dqn', False)),
)
agent.online_net.load_state_dict(ckpt['online_net'])
agent.online_net.eval()

N_EPISODES = 10
returns = []
for ep in range(N_EPISODES):
    env = make_eval_env(seed=42 + ep)
    obs, _ = env.reset(seed=42 + ep)
    total, done = 0.0, False
    while not done:
        action = agent.select_action(obs, eval_mode=True)
        obs, r, term, trunc, _ = env.step(action)
        total += r; done = term or trunc
    env.close()
    returns.append(total)
    print(f'  Episode {ep+1:>2}: {total:.0f}')

print(f'\nMean ± std : {np.mean(returns):.1f} ± {np.std(returns):.1f}')
print(f'Min / Max  : {np.min(returns):.0f} / {np.max(returns):.0f}')

## Cell 6 — record videos (trained agent + random baseline)

Records 30fps MP4s for visual comparison. Bypasses `gymnasium.RecordVideo` (fps=None crash on Kaggle) and uses `imageio` directly. Set `RECORD_VIDEO = False` to skip if short on time.

In [ ]:
import os
os.chdir(REPO_DIR)

RECORD_VIDEO = True
if RECORD_VIDEO:
    vdir = os.path.join(REPO_DIR, 'videos')
    if os.path.isdir(vdir):
        for n in os.listdir(vdir):
            full = os.path.join(vdir, n)
            if os.path.isfile(full): os.remove(full)

    FINAL_CKPT = os.path.join(CKPT_DIR, exp_name, 'final.pt')
    if os.path.exists(FINAL_CKPT):
        !python scripts/record_video.py --checkpoint {FINAL_CKPT} --episodes 1 --output videos/trained.mp4
    else:
        print(f'[skip] no final.pt at {FINAL_CKPT}; skipping trained video')

    !python scripts/record_video.py --episodes 1 --output videos/random.mp4

    print('\n[videos] written to:', vdir)
    for n in sorted(os.listdir(vdir)):
        full = os.path.join(vdir, n)
        print(f'  {n} ({os.path.getsize(full):,} bytes)')
else:
    print('[videos] skipped')

## Cell 7 — zip everything for download

Keeps the last 3 step checkpoints + `final.pt`, then zips the experiment folder and videos into `results.zip`. Click the FileLink to download, or grab it from the Output tab on the right sidebar.

In [ ]:
import os, glob, shutil, zipfile
from IPython.display import FileLink

KEEP_LAST_N = 3
exp_dir = os.path.join(CKPT_DIR, exp_name)

# Prune old step checkpoints
ckpts = sorted(glob.glob(os.path.join(exp_dir, 'step_*.pt')))
for old in ckpts[:-KEEP_LAST_N]:
    os.remove(old)
print(f'Kept last {min(KEEP_LAST_N, len(ckpts))} step checkpoints + final.pt')

ZIP_PATH = '/kaggle/working/results.zip'
if os.path.exists(ZIP_PATH): os.remove(ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Checkpoints + TensorBoard logs
    for root, _, files in os.walk(exp_dir):
        for f in files:
            full = os.path.join(root, f)
            zf.write(full, arcname=os.path.relpath(full, REPO_DIR))
    # Videos (if recorded)
    vdir = os.path.join(REPO_DIR, 'videos')
    if os.path.isdir(vdir):
        for f in os.listdir(vdir):
            full = os.path.join(vdir, f)
            if os.path.isfile(full):
                zf.write(full, arcname=os.path.join('videos', f))

size_mb = os.path.getsize(ZIP_PATH) / 1e6
print(f'Archive: {ZIP_PATH} ({size_mb:.1f} MB)')
FileLink(ZIP_PATH, result_html_prefix=f'Click to download ({size_mb:.1f} MB): ')